# Bronze Data EDA

Senior data-engineering inspection notebook for the Bronze Wikimedia pageview layer.

Goals:

- Confirm the acquisition contract: plan, manifest, local files, and 72-hour scope agree.
- Detect storage issues early: missing files, extra files, duplicate manifest rows, size mismatches, corrupt gzip files, and stale paths.
- Profile raw content without mutating Bronze data.
- Produce operational signals that tell us whether it is safe to build or rebuild Silver.

This notebook is read-only. Expensive full scans are behind explicit toggles.


## 1. Setup


In [ ]:
from __future__ import annotations

import gzip
import hashlib
import json
import re
import sys
from collections import Counter
from itertools import islice
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from wikitrend.pageviews import DEFAULT_SOURCE_PROJECTS, PROJECT_CODE_MAP, parse_dump_filename, parse_pageview_line
from wikitrend.storage import raw_file_path

RAW_DIR = ROOT / "data" / "raw" / "pageviews"
MANIFEST_PATH = ROOT / "data" / "raw" / "pageviews_manifest.json"
PLAN_PATH = ROOT / "configs" / "pageview_download_plan.json"

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8-sig"))

def bytes_to_gb(value: int | float) -> float:
    return round(float(value) / 1_000_000_000, 3)

assert RAW_DIR.exists(), f"Missing raw directory: {RAW_DIR}"
assert MANIFEST_PATH.exists(), f"Missing manifest: {MANIFEST_PATH}"
assert PLAN_PATH.exists(), f"Missing acquisition plan: {PLAN_PATH}"


## 2. Acquisition Contract


In [ ]:
plan = read_json(PLAN_PATH)
manifest = read_json(MANIFEST_PATH)
manifest_df = pd.DataFrame(manifest.get("files", []))

if not manifest_df.empty:
    manifest_df["timestamp_hour"] = pd.to_datetime(manifest_df["timestamp_hour"], utc=True)
    manifest_df["date"] = manifest_df["timestamp_hour"].dt.date.astype(str)
    manifest_df["hour"] = manifest_df["timestamp_hour"].dt.hour
    manifest_df["size_gb"] = manifest_df["size_bytes"] / 1_000_000_000

contract = {
    "plan_id": plan.get("plan_id"),
    "plan_window": f"{plan.get('start_date')} to {plan.get('end_date')}",
    "plan_expected_hours": plan.get("expected_hours"),
    "manifest_plan_id": manifest.get("plan_id"),
    "manifest_files": len(manifest_df),
    "manifest_first_hour": None if manifest_df.empty else manifest_df["timestamp_hour"].min(),
    "manifest_last_hour": None if manifest_df.empty else manifest_df["timestamp_hour"].max(),
    "manifest_size_gb": 0.0 if manifest_df.empty else bytes_to_gb(manifest_df["size_bytes"].sum()),
}

pd.DataFrame([contract])


In [ ]:
expected_hours = pd.date_range(
    start=pd.Timestamp(plan["start_date"], tz="UTC"),
    end=pd.Timestamp(plan["end_date"], tz="UTC") + pd.Timedelta(hours=23),
    freq="h",
)
expected_df = pd.DataFrame({"timestamp_hour": expected_hours})
expected_df["expected_filename"] = expected_df["timestamp_hour"].dt.strftime("pageviews-%Y%m%d-%H0000.gz")

expected_vs_manifest = expected_df.merge(
    manifest_df[["filename", "timestamp_hour"]] if not manifest_df.empty else pd.DataFrame(columns=["filename", "timestamp_hour"]),
    on="timestamp_hour",
    how="left",
)
expected_vs_manifest["in_manifest"] = expected_vs_manifest["filename"].notna()

scope_summary = {
    "expected_hours": len(expected_df),
    "manifest_hours": int(expected_vs_manifest["in_manifest"].sum()),
    "missing_manifest_hours": int((~expected_vs_manifest["in_manifest"]).sum()),
    "duplicate_manifest_filenames": int(manifest_df["filename"].duplicated().sum()) if not manifest_df.empty else 0,
    "duplicate_manifest_hours": int(manifest_df["timestamp_hour"].duplicated().sum()) if not manifest_df.empty else 0,
}

pd.DataFrame([scope_summary])


In [ ]:
expected_vs_manifest.loc[~expected_vs_manifest["in_manifest"]].head(25)


## 3. Local File Inventory


In [ ]:
local_files = sorted(RAW_DIR.rglob("pageviews-*.gz"))
local_df = pd.DataFrame(
    {
        "filename": [path.name for path in local_files],
        "local_path": [path.relative_to(RAW_DIR).as_posix() for path in local_files],
        "actual_size_bytes": [path.stat().st_size for path in local_files],
    }
)

inventory_df = manifest_df.merge(local_df, on="filename", how="outer", indicator=True)
if not inventory_df.empty:
    inventory_df["size_matches"] = inventory_df["size_bytes"].eq(inventory_df["actual_size_bytes"])
    inventory_df["actual_size_gb"] = inventory_df["actual_size_bytes"].fillna(0) / 1_000_000_000

inventory_summary = {
    "manifest_files": len(manifest_df),
    "local_files": len(local_df),
    "files_in_both": int((inventory_df["_merge"] == "both").sum()) if not inventory_df.empty else 0,
    "missing_local_files": int((inventory_df["_merge"] == "left_only").sum()) if not inventory_df.empty else 0,
    "extra_local_files": int((inventory_df["_merge"] == "right_only").sum()) if not inventory_df.empty else 0,
    "size_mismatches": int((inventory_df["_merge"].eq("both") & ~inventory_df["size_matches"]).sum()) if not inventory_df.empty else 0,
    "local_size_gb": bytes_to_gb(local_df["actual_size_bytes"].sum()) if not local_df.empty else 0.0,
}

pd.DataFrame([inventory_summary])


In [ ]:
inventory_df.loc[
    inventory_df["_merge"].ne("both") | ~inventory_df["size_matches"],
    ["filename", "_merge", "relative_path", "local_path", "size_bytes", "actual_size_bytes", "size_matches"],
].head(100)


## 4. Filename And Path Consistency


In [ ]:
filename_re = re.compile(r"^pageviews-(?P<yyyymmdd>\d{8})-(?P<hour>\d{2})0000\.gz$")

def filename_contract(row: pd.Series) -> dict:
    filename = str(row.get("filename", ""))
    match = filename_re.match(filename)
    if not match:
        return {"filename_valid": False, "filename_date": None, "filename_hour": None}
    raw_date = match.group("yyyymmdd")
    return {
        "filename_valid": True,
        "filename_date": f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:8]}",
        "filename_hour": int(match.group("hour")),
    }

if manifest_df.empty:
    filename_check_df = pd.DataFrame()
else:
    filename_check_df = pd.concat(
        [manifest_df, manifest_df.apply(filename_contract, axis=1, result_type="expand")],
        axis=1,
    )
    filename_check_df["date_matches_timestamp"] = filename_check_df["filename_date"].eq(filename_check_df["date"])
    filename_check_df["hour_matches_timestamp"] = filename_check_df["filename_hour"].eq(filename_check_df["hour"])
    filename_check_df["path_contains_filename"] = filename_check_df.apply(
        lambda row: str(row["relative_path"]).endswith(str(row["filename"])), axis=1
    )

filename_check_df.loc[
    ~filename_check_df.get("filename_valid", pd.Series(dtype=bool))
    | ~filename_check_df.get("date_matches_timestamp", pd.Series(dtype=bool))
    | ~filename_check_df.get("hour_matches_timestamp", pd.Series(dtype=bool))
    | ~filename_check_df.get("path_contains_filename", pd.Series(dtype=bool)),
    ["filename", "relative_path", "timestamp_hour", "filename_valid", "date_matches_timestamp", "hour_matches_timestamp", "path_contains_filename"],
].head(100)


## 5. File Size EDA


In [ ]:
if inventory_df.empty:
    print("No inventory rows available.")
else:
    size_profile = inventory_df.loc[inventory_df["_merge"].eq("both")].copy()
    display(size_profile["actual_size_bytes"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_frame("bytes"))

    hourly_sizes = size_profile.merge(
        manifest_df[["filename", "timestamp_hour", "date", "hour"]], on="filename", how="left", suffixes=("", "_manifest")
    )
    hourly_sizes = hourly_sizes.sort_values("timestamp_hour")
    display(hourly_sizes[["timestamp_hour", "filename", "actual_size_bytes", "actual_size_gb"]].head())
    display(hourly_sizes[["timestamp_hour", "filename", "actual_size_bytes", "actual_size_gb"]].tail())


In [ ]:
if not inventory_df.empty:
    size_profile = inventory_df.loc[inventory_df["_merge"].eq("both")].copy()
    if not size_profile.empty:
        q1 = size_profile["actual_size_bytes"].quantile(0.25)
        q3 = size_profile["actual_size_bytes"].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        display(
            size_profile.loc[
                size_profile["actual_size_bytes"].lt(lower) | size_profile["actual_size_bytes"].gt(upper),
                ["filename", "actual_size_bytes", "actual_size_gb"],
            ].sort_values("actual_size_bytes")
        )


In [ ]:
if not manifest_df.empty:
    by_hour = manifest_df.groupby("hour", as_index=False).agg(files=("filename", "size"), avg_size_mb=("size_bytes", lambda s: s.mean() / 1_000_000))
    display(by_hour)
    by_hour.plot(x="hour", y="avg_size_mb", kind="bar", figsize=(12, 4), title="Average Bronze gzip size by UTC hour")


## 6. Gzip And Hash Integrity


In [ ]:
VALIDATE_ALL_GZIP = False
HASH_ALL_FILES = False
SAMPLE_FILES = 8

files_to_check = local_files if VALIDATE_ALL_GZIP else local_files[:SAMPLE_FILES]


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

integrity_rows = []
for path in files_to_check:
    row = {"filename": path.name, "gzip_ok": False, "sha256_matches": None, "error": None}
    try:
        with gzip.open(path, "rb") as handle:
            while handle.read(1024 * 1024):
                pass
        row["gzip_ok"] = True
        if HASH_ALL_FILES:
            expected_hash = manifest_df.set_index("filename").get("sha256", pd.Series(dtype=object)).get(path.name)
            actual_hash = sha256_file(path)
            row["sha256_matches"] = actual_hash == expected_hash
    except Exception as exc:
        row["error"] = repr(exc)
    integrity_rows.append(row)

integrity_df = pd.DataFrame(integrity_rows)
integrity_df


## 7. Content Profile From Raw Samples


In [ ]:
MAX_FILES_TO_SAMPLE = 4
LINES_PER_FILE = 250_000
SUPPORTED = set(DEFAULT_SOURCE_PROJECTS)

sample_records = []
source_rows = []
parse_rows = []

for path in local_files[:MAX_FILES_TO_SAMPLE]:
    date_value, hour = parse_dump_filename(path.name)
    source_counter = Counter()
    supported_counter = Counter()
    malformed_supported = 0
    sampled_lines = 0
    parsed_supported = 0

    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as handle:
        for line in islice(handle, LINES_PER_FILE):
            sampled_lines += 1
            parts = line.rstrip("\n").split(" ")
            source_project = parts[0] if parts else ""
            source_counter[source_project] += 1

            if source_project not in SUPPORTED:
                continue
            supported_counter[source_project] += 1
            record = parse_pageview_line(line, date_value, hour)
            if record is None:
                malformed_supported += 1
                continue
            parsed_supported += 1
            sample_records.append(record.to_dict())

    parse_rows.append(
        {
            "filename": path.name,
            "sampled_lines": sampled_lines,
            "distinct_source_projects": len(source_counter),
            "supported_lines": sum(supported_counter.values()),
            "parsed_supported_lines": parsed_supported,
            "malformed_supported_lines": malformed_supported,
            "supported_line_rate": 0 if sampled_lines == 0 else sum(supported_counter.values()) / sampled_lines,
        }
    )
    for source_project, count in source_counter.most_common(25):
        source_rows.append({"filename": path.name, "source_project": source_project, "sample_rows": count})

sample_df = pd.DataFrame(sample_records)
parse_profile_df = pd.DataFrame(parse_rows)
source_profile_df = pd.DataFrame(source_rows)

display(parse_profile_df)
display(source_profile_df.head(50))


In [ ]:
if sample_df.empty:
    print("No supported records found in sample window.")
else:
    display(
        sample_df.groupby(["project", "access_mode"], dropna=False)
        .agg(rows=("page_title", "size"), views=("view_count", "sum"), response_size=("response_size", "sum"))
        .assign(avg_response_size_per_view=lambda df: df["response_size"] / df["views"].where(df["views"].ne(0)))
        .sort_values("views", ascending=False)
    )


In [ ]:
if not sample_df.empty:
    display(
        sample_df.sort_values("view_count", ascending=False)[
            ["date", "hour", "source_project", "project", "access_mode", "normalized_title", "view_count", "response_size"]
        ].head(50)
    )


In [ ]:
if not sample_df.empty:
    hourly_sample = (
        sample_df.groupby(["date", "hour", "project", "access_mode"], dropna=False)
        .agg(rows=("page_title", "size"), views=("view_count", "sum"))
        .reset_index()
    )
    display(hourly_sample.head(50))
    hourly_sample.pivot_table(index=["date", "hour"], columns="project", values="views", aggfunc="sum", fill_value=0).plot(
        figsize=(14, 5), title="Sampled supported-project views by hour"
    )


## 8. Optional Full Bronze Line Counts


In [ ]:
COUNT_ALL_LINES = False

if COUNT_ALL_LINES:
    line_count_rows = []
    for path in local_files:
        with gzip.open(path, "rt", encoding="utf-8", errors="replace") as handle:
            line_count = sum(1 for _ in handle)
        line_count_rows.append({"filename": path.name, "line_count": line_count})
    line_count_df = pd.DataFrame(line_count_rows)
    display(line_count_df.describe())
else:
    print("Set COUNT_ALL_LINES = True to scan every gzip file for raw line counts.")


## 9. Bronze Readiness Checklist

A Bronze layer is ready for Silver when:

- `plan_expected_hours`, `manifest_files`, and `local_files` all equal the active scope.
- Missing local files, extra local files, duplicate manifest filenames, and duplicate manifest hours are zero.
- Size mismatches are zero.
- Gzip validation passes for the sample, or for all files before an important reproducibility checkpoint.
- Supported project samples include the project/access combinations expected in Silver.
- No unexplained file-size outliers appear for the active window.
